# Train Word-Level ASL Recognition Model

This notebook trains a BiLSTM model for recognizing full ASL words from sequences of holistic landmarks.


In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import tensorflow as tf
from data.sequence_loader import SequenceDataLoader
from models.sequence_classifier import create_bilstm_classifier, save_model


In [ ]:
loader = SequenceDataLoader(data_dir='../data/wlasl_landmarks', sequence_length=30)
X, y_categorical, y = loader.load_data()

if X is None:
    print("ERROR: No data found. Please run data/wlasl_loader.py first to extract landmarks from WLASL dataset.")
else:
    print(f"\nDataset loaded successfully!")
    print(f"Total sequences: {len(X)}")
    print(f"Sequence shape: {X.shape[1:]}")
    print(f"Number of word classes: {len(loader.classes)}")
    print(f"\nClasses: {', '.join(loader.classes[:10])}..." if len(loader.classes) > 10 else f"\nClasses: {', '.join(loader.classes)}")


In [ ]:
if X is not None:
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = loader.split_data(X, y_categorical)
    
    print(f"Training sequences: {len(X_train)}")
    print(f"Validation sequences: {len(X_val)}")
    print(f"Test sequences: {len(X_test)}")


In [ ]:
if X is not None:
    model = create_bilstm_classifier(
        sequence_length=X.shape[1],
        feature_dim=X.shape[2],
        num_classes=len(loader.classes),
        lstm_units=[128, 64],
        dropout_rate=0.3
    )
    model.summary()


In [ ]:
if X is not None:
    history = model.fit(
        X_train, y_train,
        batch_size=32,
        epochs=50,
        validation_data=(X_val, y_val),
        verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
        ]
    )


In [ ]:
if X is not None:
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f}")


In [ ]:
if X is not None:
    save_model(
        model,
        '../models/word_model.json',
        '../models/word_model.h5'
    )
    
    import json
    class_mapping = {
        'classes': loader.classes,
        'class_to_idx': loader.class_to_idx,
        'idx_to_class': {int(k): v for k, v in loader.idx_to_class.items()}
    }
    with open('../models/word_class_mapping.json', 'w') as f:
        json.dump(class_mapping, f, indent=2)
    
    print("Model and class mapping saved successfully")


In [ ]:
if X is not None:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Model Accuracy')
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Model Loss')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('../models/word_training_history.png', dpi=150)
    plt.show()
